# Load and Process

In [24]:
# 📂 Load Data (Clinical & Methylation)
import pandas as pd
import numpy as np
import os
from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import train_test_split

# Suppress warnings
import warnings
warnings.filterwarnings("ignore")

# Set a random state for reproducibility
random_state = 42
# Set base path
base_path = "/Users/nijat/Downloads/OneDrive_1_2024-09-21/Data"

# Load Clinical Data
clinical_data_path = os.path.join(base_path, "2019_TCGA-CDR-SupplementalTableS1.xlsx")
main_data_df = pd.read_excel(clinical_data_path)

# Load Tumor Classification Data
new_labels_path = os.path.join(base_path, "Matrix_WHO2021.csv")
new_categories = pd.read_csv(new_labels_path)

# Load Methylation Data (GBM & LGG)
gbm_data = pd.read_csv(os.path.join(base_path, "GBM_450K_Filtered_X-Y,SNPs,Non-overlap850K.csv"))
lgg_data = pd.read_csv(os.path.join(base_path, "LGG-450K_Filtered_X-Y,SNPs,Non-overlap850K.csv"))

print("✅ Data Loaded Successfully!")

# Rename column for consistency
main_data_df.rename(columns={'bcr_patient_barcode': 'Patient_ID'}, inplace=True)

# Filter Clinical Data for GBM & LGG tumor types
filtered_data_df = main_data_df[main_data_df['type'].isin(['GBM', 'LGG'])]

# Merge Clinical and Tumor Classification Data
merged_clinical_data = pd.merge(filtered_data_df, new_categories, on='Patient_ID', how='inner')

print(f"✅ Merged Clinical Data Shape: {merged_clinical_data.shape}")

# Process GBM & LGG Data (Transpose & Rename)
def preprocess_methylation_data(df):
    df_trns = df.T  # Transpose data (genes as columns)
    df_trns.columns = df_trns.iloc[0]  # First row as column names
    return df_trns[1:]  # Remove first row

gbm_trns_final = preprocess_methylation_data(gbm_data)
lgg_trns_final = preprocess_methylation_data(lgg_data)

# Merge GBM & LGG Methylation Data
merged_methylation_data = pd.concat([gbm_trns_final, lgg_trns_final])

# Ensure "Patient_ID" is named correctly
merged_methylation_data.rename(columns={'Index': 'Patient_ID'}, inplace=True)

# Merge Methylation & Clinical Data
final_data = pd.merge(merged_methylation_data, merged_clinical_data, left_index=True, right_on='Patient_ID', how='inner')

print(f"✅ Final Merged Data Shape: {final_data.shape}")


ParserError: Error tokenizing data. C error: Calling read(nbytes) on source failed. Try engine='python'.

🔹 Step 2: Split Data (Before Feature Selection)

In [ ]:
# Select all methylation probes (excluding survival & categorical data)
methylation_features = [col for col in final_data.columns if col.startswith("cg")]
X = final_data[methylation_features]
y = final_data[['OS.time', 'OS']]  # Survival labels

# Split into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"✅ Training Set Shape: {X_train.shape}")
print(f"✅ Test Set Shape: {X_test.shape}")


✅ Training Set Shape: (522, 401122)
✅ Test Set Shape: (131, 401122)


# 🔹 Step 3: Apply Variance Thresholding (On Train Set Only)

In [ ]:
# Apply variance thresholding ONLY on training data
variance_thresh = VarianceThreshold(threshold=0.01)  # Adjust as needed
X_train_reduced = variance_thresh.fit_transform(X_train)

# Get selected feature names
selected_feature_names = X_train.columns[variance_thresh.get_support()]

# Transform test set using the same selected features
X_test_reduced = X_test[selected_feature_names]

print(f"✅ Variance Thresholding Applied: Reduced to {X_train_reduced.shape[1]} Features")


✅ Variance Thresholding Applied: Reduced to 163075 Features


# 🔹 Use Penalized (LASSO) Cox Regression

In [42]:
from sksurv.ensemble import RandomSurvivalForest
from sksurv.util import Surv
from sklearn.model_selection import RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import make_scorer
import pandas as pd
import numpy as np
from sksurv.metrics import concordance_index_censored

# ✅ Step 1: Convert Processed Data into DataFrames
X_train_reduced_df = pd.DataFrame(X_train_reduced, columns=selected_feature_names)
X_test_reduced_df = pd.DataFrame(X_test_reduced, columns=selected_feature_names)

# ✅ Step 2: Ensure Data is Numeric
X_train_reduced_df = X_train_reduced_df.apply(pd.to_numeric, errors='coerce')
X_test_reduced_df = X_test_reduced_df.apply(pd.to_numeric, errors='coerce')

# ✅ Step 3: Check & Report Missing Values
missing_values_train = X_train_reduced_df.isnull().sum().sum()
missing_values_test = X_test_reduced_df.isnull().sum().sum()
print(f"🔍 Missing Values in Training Set: {missing_values_train}")
print(f"🔍 Missing Values in Test Set: {missing_values_test}")

# ✅ Step 4: Drop any remaining missing values
X_train_reduced_df.dropna(inplace=True)
X_test_reduced_df.dropna(inplace=True)

# ✅ Step 5: Ensure `y_train` has No Missing Values
y_train = y_train.dropna(subset=["OS", "OS.time"])

# ✅ Step 6: **Find Common Indices Between `X_train_reduced_df` and `y_train`**
common_indices = X_train_reduced_df.index.intersection(y_train.index)

# ✅ Step 7: Align `X_train_reduced_df` and `y_train` to Ensure They Have the Same Indices
X_train_reduced_df = X_train_reduced_df.loc[common_indices]
y_train = y_train.loc[common_indices]

# ✅ Step 8: Standardize Features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_train_reduced_df)
X_rsf = pd.DataFrame(X_scaled, columns=selected_feature_names, index=common_indices)

# ✅ Step 9: Convert `y_train` to Survival Format
y_surv = Surv.from_dataframe("OS", "OS.time", y_train)

# ✅ Step 10: Verify Data Alignment
print(f"✅ X_rsf shape: {X_rsf.shape}")   # Should match `y_surv`
print(f"✅ y_surv shape: {y_surv.shape}")
assert X_rsf.shape[0] == y_surv.shape[0], "❌ Mismatch between X_rsf and y_surv!"

# ✅ Step 11: Define C-Index Scorer
# def concordance_index_scorer(estimator, X, y):
#     """
#     Custom Concordance Index Scorer for Hyperparameter Tuning.
#     Converts `y` back to its `OS` (event) and `OS.time` (duration) values.
#     """
#     times, events = y["OS.time"], y["OS"]
    
#     # Ensure the estimator makes predictions that align with survival times
#     risk_scores = estimator.predict(X)
    
#     # Compute the concordance index
#     cindex = concordance_index_censored(events, times, risk_scores)[0]
#     return cindex

def concordance_index_scorer(estimator, X, y):
    """
    Custom Concordance Index Scorer for Hyperparameter Tuning.
    Converts `y` back to its `OS` (event) and `OS.time` (duration) values.
    """
    times = np.array([t for t, e in y])  # Extract survival times
    events = np.array([e for t, e in y])  # Extract event indicators
    
    # Make predictions
    risk_scores = estimator.predict(X)
    
    # Compute concordance index
    cindex, _, _, _, _ = concordance_index_censored(events, times, risk_scores)
    
    return cindex



cindex_scorer = make_scorer(concordance_index_scorer, greater_is_better=True)

# ✅ Step 12: Define Hyperparameter Grid
param_grid = {
    "n_estimators": [50, 100, 150, 200],
    "max_depth": [3, 5, 7, 10],
    "min_samples_split": [5, 10, 15],
    "min_samples_leaf": [2, 5, 10],
}

# ✅ Step 13: Perform Randomized Search with C-Index Scoring
rsf = RandomSurvivalForest(random_state=42, n_jobs=-1, verbose=1)

rsf_search = RandomizedSearchCV(
    rsf, param_grid, cv=3, scoring=cindex_scorer, n_iter=10, random_state=42, n_jobs=-1, verbose=2
)

print("🔍 Hyperparameter tuning in progress...")
rsf_search.fit(X_rsf, y_surv)  # ✅ Verbose output enabled

# ✅ Step 14: Get Best Parameters
best_params = rsf_search.best_params_
print(f"✅ Best Hyperparameters Found: {best_params}")

# ✅ Step 15: Train RSF with Best Parameters and Verbose Output
best_rsf = rsf_search.best_estimator_
print("🚀 Training RSF with best parameters...")
best_rsf.fit(X_rsf, y_surv)

# ✅ Step 16: Evaluate Performance
c_index = best_rsf.score(X_rsf, y_surv)
print(f"✅ C-Index Score: {c_index:.4f}")


🔍 Missing Values in Training Set: 0
🔍 Missing Values in Test Set: 0
✅ X_rsf shape: (64, 163075)
✅ y_surv shape: (64,)
🔍 Hyperparameter tuning in progress...
Fitting 3 folds for each of 10 candidates, totalling 30 fits


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 14 concurrent workers.
[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 14 concurrent workers.
[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 14 concurrent workers.
[Parallel(n_jobs=-1)]: Done  22 tasks      | elapsed:    0.1s
[Parallel(n_jobs=-1)]: Done  22 tasks      | elapsed:    0.1s
[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 14 concurrent workers.
[Parallel(n_jobs=-1)]: Done  22 tasks      | elapsed:    0.1s
[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 14 concurrent workers.
[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 14 concurrent workers.
[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 14 concurrent workers.
[Parallel(n_jobs=-1)]: Done  22 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done  22 tasks      | elapsed:    0.2s
[Parallel(n_jobs=-1)]: Done  22 tasks      | elapsed:    0.2s
[Parallel(n_jobs=-1)]: Done  22 tasks      | elapsed: 

[CV] END max_depth=10, min_samples_leaf=2, min_samples_split=15, n_estimators=100; total time=   1.4s
[CV] END max_depth=10, min_samples_leaf=2, min_samples_split=15, n_estimators=100; total time=   1.5s
[CV] END max_depth=10, min_samples_leaf=2, min_samples_split=15, n_estimators=100; total time=   1.5s
[CV] END max_depth=3, min_samples_leaf=5, min_samples_split=10, n_estimators=200; total time=   1.5s
[CV] END max_depth=3, min_samples_leaf=5, min_samples_split=10, n_estimators=200; total time=   1.5s
[CV] END max_depth=7, min_samples_leaf=10, min_samples_split=5, n_estimators=100; total time=   1.4s
[CV] END max_depth=7, min_samples_leaf=2, min_samples_split=15, n_estimators=150; total time=   1.5s
[CV] END max_depth=3, min_samples_leaf=5, min_samples_split=10, n_estimators=200; total time=   1.5s
[CV] END max_depth=7, min_samples_leaf=10, min_samples_split=5, n_estimators=100; total time=   1.4s
[CV] END max_depth=7, min_samples_leaf=10, min_samples_split=5, n_estimators=100; total 

[Parallel(n_jobs=14)]: Using backend ThreadingBackend with 14 concurrent workers.
[Parallel(n_jobs=14)]: Using backend ThreadingBackend with 14 concurrent workers.
[Parallel(n_jobs=14)]: Done  22 tasks      | elapsed:    0.0s
[Parallel(n_jobs=14)]: Done  22 tasks      | elapsed:    0.0s
[Parallel(n_jobs=14)]: Done 150 out of 150 | elapsed:    0.0s finished
[Parallel(n_jobs=14)]: Done 150 out of 150 | elapsed:    0.0s finished
/opt/anaconda3/envs/glioma/lib/python3.11/site-packages/sklearn/model_selection/_validation.py:960: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/opt/anaconda3/envs/glioma/lib/python3.11/site-packages/sklearn/model_selection/_validation.py", line 949, in _score
    scores = scorer(estimator, X_test, y_test, **score_params)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/glioma/lib/python3.11/site-packages/s

[CV] END max_depth=7, min_samples_leaf=2, min_samples_split=15, n_estimators=150; total time=   1.6s
[CV] END max_depth=7, min_samples_leaf=2, min_samples_split=15, n_estimators=150; total time=   1.6s


...

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 14 concurrent workers.
[Parallel(n_jobs=-1)]: Done  22 tasks      | elapsed:    0.1s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:    0.2s finished


✅ Best Hyperparameters Found: {'n_estimators': 100, 'min_samples_split': 15, 'min_samples_leaf': 2, 'max_depth': 10}
🚀 Training RSF with best parameters...


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 14 concurrent workers.
[Parallel(n_jobs=-1)]: Done  22 tasks      | elapsed:    0.1s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:    0.2s finished


✅ C-Index Score: 0.9306


[Parallel(n_jobs=14)]: Using backend ThreadingBackend with 14 concurrent workers.
[Parallel(n_jobs=14)]: Done  22 tasks      | elapsed:    0.0s
[Parallel(n_jobs=14)]: Done 100 out of 100 | elapsed:    0.0s finished


In [43]:
# ✅ Print the best hyperparameters
print(f"✅ Best Hyperparameters Found: {rsf_search.best_params_}")

# ✅ Print the best estimator (model with the best params)
print(f"✅ Best Estimator: {rsf_search.best_estimator_}")

# ✅ Print the best score (C-Index from the best model)
print(f"✅ Best C-Index Score: {rsf_search.best_score_:.4f}")


✅ Best Hyperparameters Found: {'n_estimators': 100, 'min_samples_split': 15, 'min_samples_leaf': 2, 'max_depth': 10}
✅ Best Estimator: RandomSurvivalForest(max_depth=10, min_samples_leaf=2, min_samples_split=15,
                     n_jobs=-1, random_state=42, verbose=1)
✅ Best C-Index Score: nan


In [44]:
best_rsf = rsf_search.best_estimator_
best_rsf.fit(X_rsf, y_surv)


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 14 concurrent workers.
[Parallel(n_jobs=-1)]: Done  22 tasks      | elapsed:    0.1s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:    0.2s finished


RandomSurvivalForest(max_depth=10, min_samples_leaf=2, min_samples_split=15,
                     n_jobs=-1, random_state=42, verbose=1)

In [45]:
y_test_surv = Surv.from_dataframe("OS", "OS.time", y_test)  # Convert y_test
test_preds = best_rsf.predict(X_test_reduced_df)
test_cindex, _, _, _, _ = concordance_index_censored(y_test_surv["OS"], y_test_surv["OS.time"], test_preds)

print(f"🚀 Test C-Index: {test_cindex:.4f}")


🚀 Test C-Index: 0.1807


[Parallel(n_jobs=14)]: Using backend ThreadingBackend with 14 concurrent workers.
[Parallel(n_jobs=14)]: Done  22 tasks      | elapsed:    0.0s
[Parallel(n_jobs=14)]: Done 100 out of 100 | elapsed:    0.0s finished
